In [1]:
from flask import Flask, render_template, request, jsonify
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [4]:
# MEMBACA DATASET
df = pd.read_csv("diabetes_raw.csv")
print(df.head())

     id   chol  stab.glu   hdl  ratio  glyhb    location  age  gender  height  \
0  1000  203.0        82  56.0    3.6   4.31  Buckingham   46  female    62.0   
1  1001  165.0        97  24.0    6.9   4.44  Buckingham   29  female    64.0   
2  1002  228.0        92  37.0    6.2   4.64  Buckingham   58  female    61.0   
3  1003   78.0        93  12.0    6.5   4.63  Buckingham   67    male    67.0   
4  1005  249.0        90  28.0    8.9   7.72  Buckingham   64    male    68.0   

   weight   frame  bp.1s  bp.1d  bp.2s  bp.2d  waist   hip  time.ppn  
0   121.0  medium  118.0   59.0    NaN    NaN   29.0  38.0     720.0  
1   218.0   large  112.0   68.0    NaN    NaN   46.0  48.0     360.0  
2   256.0   large  190.0   92.0  185.0   92.0   49.0  57.0     180.0  
3   119.0   large  110.0   50.0    NaN    NaN   33.0  38.0     480.0  
4   183.0  medium  138.0   80.0    NaN    NaN   44.0  41.0     300.0  


In [15]:
# KLASIFIKASI TARGET DARI KADAR GULA DARAH (glyhb)
# Tidak (<5.7), Terkena (>=5.7)
def label_glyhb(val):
    if val < 5.7:
        return "Tidak"
    else:
        return "Terkena"
df['target'] = df['glyhb'].apply(label_glyhb)
print(df[['glyhb', 'target']].head())

   glyhb   target
0   4.31    Tidak
1   4.44    Tidak
2   4.64    Tidak
3   4.63    Tidak
4   7.72  Terkena


In [14]:
# FITUR (jangan gunakan glyhb sebagai fitur)
features = ['stab.glu', 'weight', 'waist', 'hip']
full = df[features + ['target']].dropna()
X = full[features]
y = full['target']
print(X.head())
print(y.head())

   stab.glu  weight  waist   hip
0        82   121.0   29.0  38.0
1        97   218.0   46.0  48.0
2        92   256.0   49.0  57.0
3        93   119.0   33.0  38.0
4        90   183.0   44.0  41.0
0      Tidak
1      Tidak
2      Tidak
3      Tidak
4    Terkena
Name: target, dtype: str


In [16]:
# TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Train set size: 320 samples
Test set size: 80 samples


In [17]:
# SCALING
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(scaler.mean_)
print(scaler.scale_)

[106.11875  178.79375   38.103125  43.25625 ]
[51.176456   42.20627869  5.79859813  5.82585495]


In [9]:
# MODEL RANDOM FOREST
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [13]:
# EVALUASI
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='Terkena')
recall = recall_score(y_test, y_pred, pos_label='Terkena')
f1 = f1_score(y_test, y_pred, pos_label='Terkena')
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.8875
Precision: 0.9231
Recall: 0.6000
F1 Score: 0.7273


In [ ]:
app = Flask(__name__)

@app.route("/")
def home():
    return render_template('index.html')

In [ ]:
@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.json

        if data is None:
            return jsonify({'error': 'Request body harus berupa JSON'}), 400

        required = ['glucose', 'weight', 'waist', 'hip']
        for field in required:
            if field not in data or data[field] is None or data[field] == '':
                return jsonify({'error': f'Field "{field}" wajib diisi'}), 400

            try:
                val = float(data[field])
            except (ValueError, TypeError):
                return jsonify({'error': f'Field "{field}" harus berupa angka'}), 400

            if val <= 0:
                return jsonify({'error': f'Field "{field}" harus lebih dari 0'}), 400

        glucose = float(data['glucose'])
        weight_kg = float(data['weight'])
        weight_lbs = weight_kg * 2.20462
        waist_cm = float(data['waist'])
        waist_in = waist_cm / 2.54
        hip_cm = float(data['hip'])
        hip_in = hip_cm / 2.54

        new_data = pd.DataFrame({
            'stab.glu': [glucose],
            'weight': [weight_lbs],
            'waist': [waist_in],
            'hip': [hip_in]
        })

        new_scaled = scaler.transform(new_data)
        prediction = model.predict(new_scaled)[0]
        proba = model.predict_proba(new_scaled)[0]
        confidence = round(max(proba) * 100, 1)

        return jsonify({
            'classification': prediction,
            'confidence': confidence,
            'accuracy': round(accuracy * 100, 2),
            'precision': round(precision * 100, 2),
            'recall': round(recall * 100, 2),
            'f1_score': round(f1 * 100, 2)
        })

    except Exception as e:
        return jsonify({'error': f'Terjadi kesalahan: {str(e)}'}), 500

if __name__ == "__main__":
    app.run(debug=True)